# Lab 6: Fine-Tune with LoRA

The question I'm working on: if I only train a low-rank update on two projections per layer, how much of full fine-tuning do I keep, and how much memory do I actually save? I'm using SmolLM2-135M on the emotion task. Every comparison has to be controlled, and every claim I make has to come from a number in this notebook.

**Before running:** Runtime > Change runtime type > T4 GPU.

## Colab setup

In [11]:
!pip install -q -U transformers peft datasets

In [12]:
# Colab ships torchao 0.10, which the new peft refuses to load next to; this lab never uses torchao
!pip uninstall -y -q torchao

In [13]:
import os

# lab06_starter.py sits next to this notebook. On Colab, pick it in the upload box if it isn't there yet.
if not os.path.exists("lab06_starter.py"):
    from google.colab import files
    files.upload()
print("starter present:", os.path.exists("lab06_starter.py"))

starter present: True


### Does the starter run as given?

Before I change anything I want to know the environment works, so I run the starter once, unmodified. If it fails here, nothing I build later can be trusted.

In [14]:
!python lab06_starter.py

device: cuda
Loading weights: 100% 272/272 [00:00<00:00, 809.80it/s]
example prompt: 'Text: i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is\nEmotion:' answer: ' joy'
zero-shot accuracy (chance 0.167): 0.35
Loading weights: 100% 272/272 [00:00<00:00, 415.03it/s]
LoRA r=8, 30 steps: accuracy 0.510, trainable 460,800 of 134,975,808, 0.32 s/step


The starter runs on the T4 as given. Zero-shot is 0.35 on its 100-example check and LoRA r=8 reaches 0.51 after 30 steps, so the environment works before I change anything.

In [15]:
import copy
import gc
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
import datasets

import lab06_starter as S

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__)
print("device:", S.DEVICE, "|", torch.cuda.get_device_name(0) if S.DEVICE == "cuda" else "no GPU")

STEPS = 120
BATCH_SIZE = 8
SEEDS = [0, 1, 2]
RANKS = [1, 4, 16]
N_TRAIN, N_TEST, N_VALIDATION = 600, 500, 300
os.makedirs("figures", exist_ok=True)
results = {"steps": STEPS, "batch_size": BATCH_SIZE, "device": torch.cuda.get_device_name(0) if S.DEVICE == "cuda" else "cpu"}

torch 2.14.1+cu130 | transformers 5.22.2 | peft 0.21.2 | datasets 5.1.0
device: cuda | Tesla T4


## Zero-shot baseline and answer-only loss

Any gain from fine-tuning is meaningless until I know where the model starts. I'm going to score the pretrained model on the test set with no training at all, then put that number next to two floors:

- **Chance:** 1/6, if it guessed uniformly.
- **Majority label:** always answering the most common emotion. The emotion data is imbalanced, so this floor is higher than chance, and a model that beats chance can still lose to it.

I'm also counting what the model predicts, because an accuracy above chance can hide a model that says the same word every time.

In [16]:
base_model, tokenizer = S.load_model()
train_examples, test_examples, label_ids = S.load_task("emotion", n_train=N_TRAIN, n_test=N_TEST, tokenizer=tokenizer)
label_words = S.TASKS["emotion"]["words"]

print("label tokens:", {word: tokenizer.decode([token_id]) for word, token_id in zip(label_words, label_ids)})
print("example prompt:", repr(tokenizer.decode(train_examples[0]["prompt_ids"])),
      "answer:", repr(tokenizer.decode([train_examples[0]["answer_id"]])))


@torch.no_grad()
def predict_labels(model, examples, batch_size=32):
    """Same scoring as S.evaluate, but returns the predicted label for every example."""
    model.eval()
    label_tensor = torch.tensor(label_ids, device=S.DEVICE)
    predictions = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start + batch_size]
        input_ids, attention_mask, prompt_lens = S.pad_batch(chunk, tokenizer.pad_token_id)
        logits = model(input_ids=input_ids.to(S.DEVICE), attention_mask=attention_mask.to(S.DEVICE)).logits
        last_prompt_scores = logits[torch.arange(len(chunk)), (prompt_lens - 1).to(S.DEVICE)]
        predictions += last_prompt_scores[:, label_tensor].argmax(dim=-1).tolist()
    return predictions


zero_shot_accuracy = S.evaluate(base_model, tokenizer, test_examples, label_ids)
zero_shot_predictions = predict_labels(base_model, test_examples)
test_label_counts = pd.Series([label_words[example["label"]] for example in test_examples]).value_counts()
zero_shot_counts = pd.Series([label_words[prediction] for prediction in zero_shot_predictions]).value_counts()
chance = 1 / len(label_ids)
majority_rate = test_label_counts.iloc[0] / len(test_examples)

print(f"zero-shot accuracy: {zero_shot_accuracy:.3f}")
print(f"chance: {chance:.3f} | always '{test_label_counts.index[0]}': {majority_rate:.3f}")
print(pd.DataFrame({"true labels": test_label_counts, "zero-shot predictions": zero_shot_counts}).fillna(0).astype(int))
results.update(zero_shot_accuracy=zero_shot_accuracy, chance=chance, majority_label=test_label_counts.index[0],
               majority_rate=float(majority_rate))

label tokens: {'sadness': ' sadness', 'joy': ' joy', 'love': ' love', 'anger': ' anger', 'fear': ' fear', 'surprise': ' surprise'}
example prompt: 'Text: i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is\nEmotion:' answer: ' joy'
zero-shot accuracy: 0.326
chance: 0.167 | always 'joy': 0.330
          true labels  zero-shot predictions
anger              66                     88
fear               58                      5
joy               165                     32
love               39                    265
sadness           153                    110
surprise           19                      0


Zero-shot lands at 0.326. That's about twice chance (0.167), but it's level with always answering joy (0.330), so on its own the pretrained model has no usable skill here. The prediction counts show why:

- **Love:** it says love for 265 of 500 tweets, when only 39 are love.
- **Joy:** the most common label (165) gets only 32 predictions.
- **Surprise:** never predicted at all.

Most of what it gets right comes from sadness and anger. So every fine-tuned number below has to clear 0.33, not just 0.167.

### Scoring only the answer

The starter's loss scores every token in the prompt, but the test only ever looks at one thing: the label score at the last prompt position. I want the loss to train exactly that. I'll write `answer_only_labels` so only the answer token is scored. Then I'll prove it with assertions instead of trusting it:

- **One per row:** exactly one position is scored in every example.
- **No prompt:** no prompt token is scored.
- **Right token:** the scored token is the answer.

The mistake the brief warns about is scoring one position too early, and it runs without any error. So I'm also writing that broken version on purpose to make sure my checks catch it.

In [17]:
def answer_only_labels(input_ids, attention_mask, prompt_lens):
    """Score only the answer token. Everything else, prompt and padding, is -100."""
    labels = torch.full_like(input_ids, -100)
    rows = torch.arange(input_ids.shape[0])
    labels[rows, prompt_lens] = input_ids[rows, prompt_lens]
    return labels


def off_by_one_labels(input_ids, attention_mask, prompt_lens):
    """The silent mistake from the brief: scores the last prompt token instead of the answer."""
    labels = torch.full_like(input_ids, -100)
    rows = torch.arange(input_ids.shape[0])
    labels[rows, prompt_lens - 1] = input_ids[rows, prompt_lens - 1]
    return labels


def check_answer_only(labels_fn, batch):
    input_ids, attention_mask, prompt_lens = S.pad_batch(batch, tokenizer.pad_token_id)
    labels = labels_fn(input_ids, attention_mask, prompt_lens)
    scored_per_row = (labels != -100).sum(dim=1)
    assert torch.all(scored_per_row == 1), f"scored positions per row: {scored_per_row.tolist()}"
    for row, example in enumerate(batch):
        assert torch.all(labels[row, :prompt_lens[row]] == -100), f"row {row} scores a prompt token"
        assert labels[row, prompt_lens[row]].item() == example["answer_id"], f"row {row} does not score the answer"
    return labels


check_batch = train_examples[:8]
input_ids, attention_mask, prompt_lens = S.pad_batch(check_batch, tokenizer.pad_token_id)
print("prompt lengths in the check batch:", prompt_lens.tolist())

answer_labels = check_answer_only(answer_only_labels, check_batch)
print("answer_only_labels passed: one scored position per row, no prompt token scored")
print("scored tokens:", [tokenizer.decode([token_id]) for token_id in answer_labels[answer_labels != -100].tolist()])

try:
    check_answer_only(off_by_one_labels, check_batch)
except AssertionError as error:
    shifted_labels = off_by_one_labels(input_ids, attention_mask, prompt_lens)
    shifted_tokens = [tokenizer.decode([token_id]) for token_id in shifted_labels[shifted_labels != -100].tolist()]
    print("off-by-one version fails:", error, "| it scores", shifted_tokens[:3])

all_token_labels = S.all_tokens_labels(input_ids, attention_mask, prompt_lens)
print("starter's all_tokens_labels scores per row:", (all_token_labels != -100).sum(dim=1).tolist())

prompt lengths in the check batch: [38, 21, 27, 11, 14, 29, 38, 31]
answer_only_labels passed: one scored position per row, no prompt token scored
scored tokens: [' joy', ' joy', ' love', ' sadness', ' joy', ' joy', ' fear', ' joy']
off-by-one version fails: row 0 scores a prompt token | it scores [':', ':', ':']
starter's all_tokens_labels scores per row: [39, 22, 28, 12, 15, 30, 39, 32]


Every row scores exactly one token, and it's always a label word (joy, love, sadness, fear). The off-by-one version trips the prompt assertion on the first row, because it scores the `:` at the end of `Emotion:`. The starter's version scores 12 to 39 tokens per row, so in its loss the answer is one token among dozens.

The assertions show which position gets the label, but not that Hugging Face's internal shift lines it up with the position `S.evaluate` reads. To close that gap I'm working out the cross-entropy at the last prompt position myself and comparing it with the loss the model reports. If the two don't match, training and evaluation are looking at different positions.

In [18]:
with torch.no_grad():
    batch_ids, batch_mask = input_ids.to(S.DEVICE), attention_mask.to(S.DEVICE)
    model_output = base_model(input_ids=batch_ids, attention_mask=batch_mask, labels=answer_labels.to(S.DEVICE))
    last_prompt_logits = model_output.logits[torch.arange(len(check_batch)), (prompt_lens - 1).to(S.DEVICE)]
    answer_tokens = torch.tensor([example["answer_id"] for example in check_batch], device=S.DEVICE)
    manual_loss = nn.functional.cross_entropy(last_prompt_logits, answer_tokens)
print(f"HF loss {model_output.loss.item():.6f} | manual loss at last prompt position {manual_loss.item():.6f}")
assert torch.allclose(model_output.loss, manual_loss, atol=1e-5)

HF loss 6.229302 | manual loss at last prompt position 6.229302


Both losses come out at 6.229302. After its shift, Hugging Face predicts my label from the last prompt position, the same position `S.evaluate` reads, so training and evaluation are looking at the same place.

**What the starter's all-token loss actually trains:** it puts most of the gradient into predicting the "Text: … Emotion:" template and the tweet itself, which is language modelling on tweets. That isn't the skill I'm measuring, because the test never checks how well the model predicts the prompt, only which of six label tokens it scores highest at the end.